# 优化基础与梯度下降

本notebook介绍深度学习优化的基础概念,包括凸性、梯度下降及其变体。

## 学习目标

- 理解优化与深度学习的关系
- 掌握凸性的定义与性质
- 实现梯度下降(GD)算法
- 理解随机梯度下降(SGD)的优势
- 学习小批量SGD的实践技巧

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
from mpl_toolkits.mplot3d import Axes3D

torch.manual_seed(42)
np.random.seed(42)

## 1. 优化与深度学习

### 1.1 目标的区别

**优化**: 最小化目标函数(通常是训练损失)
$$
\min_{\mathbf{w}} L(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n \ell(f(\mathbf{x}_i; \mathbf{w}), y_i)
$$

**深度学习**: 最小化泛化误差(测试损失)
$$
\min_{\mathbf{w}} \mathbb{E}_{(\mathbf{x}, y) \sim p_{\text{data}}} [\ell(f(\mathbf{x}; \mathbf{w}), y)]
$$

**关键差异**:
- 优化关注**训练误差**(经验风险)
- 深度学习关注**泛化误差**(真实风险)
- 过度优化训练误差 → 过拟合

### 1.2 经验风险 vs 真实风险

用简单函数演示:

In [ ]:
def f(x):
    """真实风险(平滑)"""
    return x * torch.cos(np.pi * x)

def g(x):
    """经验风险(有噪声)"""
    return f(x) + 0.2 * torch.cos(5 * np.pi * x)

x = torch.arange(0.5, 1.5, 0.01)
plt.figure(figsize=(10, 5))
plt.plot(x, f(x), label='True Risk (Generalization)', linewidth=2)
plt.plot(x, g(x), label='Empirical Risk (Training)', linewidth=2, alpha=0.7)
plt.axvline(1.0, color='r', linestyle='--', alpha=0.5, label='Min Empirical Risk')
plt.axvline(1.1, color='g', linestyle='--', alpha=0.5, label='Min True Risk')
plt.xlabel('Parameter θ')
plt.ylabel('Risk')
plt.legend()
plt.title('训练误差最小 ≠ 泛化误差最小')
plt.grid(True, alpha=0.3)
plt.show()

print("观察: 经验风险最小值和真实风险最小值不在同一位置!")
print("这就是为什么我们需要正则化、early stopping等技巧")

### 1.3 深度学习优化的挑战

**1. 局部最小值(Local Minima)**:
- 梯度为0但不是全局最优
- 高维空间中大量存在
- 实践中不是主要问题(局部最小值往往也足够好)

**2. 鞍点(Saddle Points)**:
- 某些方向是最小值,其他方向是最大值
- 比局部最小值更常见!
- 梯度为0,优化容易卡住

**3. 梯度消失(Vanishing Gradients)**:
- 梯度接近0,参数更新极慢
- sigmoid/tanh激活函数的饱和区
- ReLU的引入部分缓解了这个问题

In [ ]:
# 可视化优化挑战
fig = plt.figure(figsize=(15, 4))

# 1. 局部最小值
ax1 = fig.add_subplot(131)
x = torch.arange(-1.0, 2.0, 0.01)
y = x * torch.cos(np.pi * x)
ax1.plot(x, y, linewidth=2)
ax1.scatter([-0.3], [f(torch.tensor(-0.3))], color='orange', s=100, zorder=5, label='Local Minimum')
ax1.scatter([1.1], [f(torch.tensor(1.1))], color='red', s=100, zorder=5, label='Global Minimum')
ax1.set_title('局部最小值')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 鞍点
ax2 = fig.add_subplot(132)
x = torch.arange(-2.0, 2.0, 0.01)
y = x**3
ax2.plot(x, y, linewidth=2)
ax2.scatter([0], [0], color='purple', s=100, zorder=5, label='Saddle Point')
ax2.set_title('鞍点 (x³)')
ax2.set_xlabel('x')
ax2.set_ylabel('f(x)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 梯度消失
ax3 = fig.add_subplot(133)
x = torch.arange(-2.0, 5.0, 0.01)
y = torch.tanh(x)
ax3.plot(x, y, linewidth=2, label='tanh(x)')
ax3.axvline(4, color='r', linestyle='--', alpha=0.5)
ax3.text(4.1, 0, 'Vanishing\nGradient\nRegion', fontsize=10)
ax3.set_title('梯度消失 (tanh)')
ax3.set_xlabel('x')
ax3.set_ylabel('f(x)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n优化挑战总结:")
print("1. 局部最小值: 梯度为0但不是全局最优")
print("2. 鞍点: 在x³的x=0处,既不是最小也不是最大")
print("3. 梯度消失: tanh在x=4处梯度≈0.0013,更新极慢")

## 2. 凸性(Convexity)

凸优化是优化理论的基石,尽管深度学习问题通常非凸,理解凸性仍然非常重要。

### 2.1 凸集(Convex Set)

集合 $\mathcal{X}$ 是凸集,如果对于任意 $\mathbf{a}, \mathbf{b} \in \mathcal{X}$ 和 $\lambda \in [0, 1]$:
$$
\lambda \mathbf{a} + (1-\lambda) \mathbf{b} \in \mathcal{X}
$$

**直观理解**: 连接集合中任意两点的线段完全在集合内部。

### 2.2 凸函数(Convex Function)

函数 $f: \mathcal{X} \to \mathbb{R}$ 是凸函数,如果:
$$
f(\lambda \mathbf{x} + (1-\lambda) \mathbf{y}) \leq \lambda f(\mathbf{x}) + (1-\lambda) f(\mathbf{y})
$$

**几何意义**: 函数图像上任意两点连线位于函数图像上方。

### 2.3 判断凸性

**一阶条件**(梯度):
$$
f(\mathbf{y}) \geq f(\mathbf{x}) + \nabla f(\mathbf{x})^\top (\mathbf{y} - \mathbf{x})
$$

**二阶条件**(Hessian):
- 凸函数: Hessian矩阵半正定 ($\mathbf{H} \succeq 0$)
- 严格凸: Hessian矩阵正定 ($\mathbf{H} \succ 0$)

In [ ]:
# 可视化凸函数 vs 非凸函数
def plot_convexity():
    x = torch.arange(-2, 2, 0.01)
    
    # 三个函数
    f_convex = 0.5 * x**2  # 凸函数
    f_nonconvex = torch.cos(np.pi * x)  # 非凸函数
    f_exp = torch.exp(0.5 * x)  # 凸函数(指数)
    
    # 选择两个点测试凸性
    segment_x = torch.tensor([-1.5, 1.0])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    functions = [
        (f_convex, '0.5x² (凸)', 'green'),
        (f_nonconvex, 'cos(πx) (非凸)', 'red'),
        (f_exp, 'e^(0.5x) (凸)', 'blue')
    ]
    
    for ax, (func, title, color) in zip(axes, functions):
        ax.plot(x, func, linewidth=2, label='f(x)')
        
        # 绘制两点之间的线段
        segment_y = torch.tensor([func[torch.abs(x - segment_x[0]).argmin()],
                                   func[torch.abs(x - segment_x[1]).argmin()]])
        ax.plot(segment_x, segment_y, 'o-', color=color, linewidth=2, 
                markersize=8, label='Linear Interpolation')
        
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('f(x)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_convexity()

print("\n凸性判断:")
print("- 左图: 线段在曲线上方 → 凸函数")
print("- 中图: 线段穿过曲线 → 非凸函数")
print("- 右图: 线段在曲线上方 → 凸函数")

### 2.4 凸函数的性质

**重要性质**:
1. **局部最小值 = 全局最小值**
2. **所有局部最小值构成凸集**
3. **凸函数之和仍是凸函数**
4. **凸函数与非负数相乘仍是凸函数**

**詹森不等式**(Jensen's Inequality):
$$
f\left(\mathbb{E}[\mathbf{X}]\right) \leq \mathbb{E}[f(\mathbf{X})]
$$

**应用**: 证明交叉熵损失是凸的,证明KL散度非负等。

In [ ]:
# 演示: 凸函数的局部最小值就是全局最小值
def demonstrate_convex_optimum():
    x = torch.arange(-3, 3, 0.01)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 凸函数: 只有一个最小值
    y_convex = x**2 + 2
    axes[0].plot(x, y_convex, linewidth=2)
    axes[0].scatter([0], [2], color='red', s=150, zorder=5, label='唯一最小值')
    axes[0].set_title('凸函数: x² + 2')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 非凸函数: 多个局部最小值
    y_nonconvex = torch.sin(x) + 0.1 * x**2
    axes[1].plot(x, y_nonconvex, linewidth=2)
    # 手动标记几个局部最小值
    local_minima_x = torch.tensor([-2.8, -0.5, 1.8])
    local_minima_y = torch.sin(local_minima_x) + 0.1 * local_minima_x**2
    axes[1].scatter(local_minima_x, local_minima_y, color='orange', s=150, 
                    zorder=5, label='局部最小值')
    axes[1].set_title('非凸函数: sin(x) + 0.1x²')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('f(x)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

demonstrate_convex_optimum()

print("\n关键区别:")
print("- 凸函数: 从任何起点出发,梯度下降都能找到全局最优")
print("- 非凸函数: 可能陷入局部最优,需要多次随机初始化")

## 3. 梯度下降(Gradient Descent)

### 3.1 泰勒展开与梯度下降

考虑函数 $f: \mathbb{R} \to \mathbb{R}$,在点 $x$ 处的一阶泰勒展开:
$$
f(x + \epsilon) \approx f(x) + \epsilon f'(x)
$$

选择 $\epsilon = -\eta f'(x)$ (负梯度方向):
$$
f(x - \eta f'(x)) \approx f(x) - \eta [f'(x)]^2 \leq f(x)
$$

**更新规则**:
$$
x \leftarrow x - \eta \nabla f(x)
$$

其中 $\eta > 0$ 是**学习率**(learning rate)。

### 3.2 一维梯度下降实现

In [ ]:
def f(x):
    """目标函数: x²"""
    return x ** 2

def f_grad(x):
    """梯度: 2x"""
    return 2 * x

def gradient_descent_1d(lr, num_iters=10, x_init=10.0):
    """一维梯度下降"""
    x = x_init
    trajectory = [x]
    
    for i in range(num_iters):
        grad = f_grad(x)
        x = x - lr * grad
        trajectory.append(x)
        if (i + 1) % 2 == 0:
            print(f"Iter {i+1}: x = {x:.4f}, f(x) = {f(x):.4f}, grad = {grad:.4f}")
    
    return trajectory

# 测试不同学习率
learning_rates = [0.05, 0.2, 1.1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lr in zip(axes, learning_rates):
    trajectory = gradient_descent_1d(lr, num_iters=10)
    
    # 绘制函数曲线
    x_range = torch.arange(-12, 12, 0.1)
    ax.plot(x_range, f(x_range), 'b-', linewidth=1, alpha=0.5, label='f(x)=x²')
    
    # 绘制优化轨迹
    trajectory_tensor = torch.tensor(trajectory)
    ax.plot(trajectory_tensor, f(trajectory_tensor), 'ro-', 
            markersize=6, linewidth=2, label=f'lr={lr}')
    
    ax.set_title(f'学习率 = {lr}')
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n学习率的影响:")
print("- lr=0.05: 收敛太慢,10步后仍未到达最优")
print("- lr=0.2: 合适,快速收敛到最优解")
print("- lr=1.1: 过大,发散!")

### 3.3 学习率的选择

**学习率太小**:
- 收敛速度慢
- 需要更多迭代
- 计算成本高

**学习率太大**:
- 可能振荡
- 甚至发散
- 无法收敛到最优

**实践建议**:
- 从小学习率开始尝试(如1e-3)
- 观察loss曲线调整
- 使用学习率调度(后续章节)

### 3.4 多维梯度下降

In [ ]:
def f_2d(x1, x2):
    """二维目标函数: x1² + 2x2²"""
    return x1**2 + 2 * x2**2

def f_2d_grad(x1, x2):
    """梯度"""
    return 2 * x1, 4 * x2

def gradient_descent_2d(lr, num_iters=20, x_init=(3.0, 3.0)):
    """二维梯度下降"""
    x1, x2 = x_init
    trajectory = [(x1, x2)]
    
    for i in range(num_iters):
        g1, g2 = f_2d_grad(x1, x2)
        x1 = x1 - lr * g1
        x2 = x2 - lr * g2
        trajectory.append((x1, x2))
    
    return trajectory

# 可视化二维梯度下降
trajectory = gradient_descent_2d(lr=0.2, num_iters=20)
trajectory = np.array(trajectory)

# 创建等高线图
x1_range = np.linspace(-4, 4, 100)
x2_range = np.linspace(-4, 4, 100)
X1, X2 = np.meshgrid(x1_range, x2_range)
Z = X1**2 + 2 * X2**2

plt.figure(figsize=(10, 8))
contour = plt.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.6)
plt.colorbar(contour, label='f(x1, x2)')
plt.plot(trajectory[:, 0], trajectory[:, 1], 'ro-', markersize=8, 
         linewidth=2, label='Optimization Path')
plt.scatter([0], [0], color='green', s=200, marker='*', 
            zorder=5, label='Optimum (0, 0)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('二维梯度下降 (f = x1² + 2x2²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"起点: ({trajectory[0, 0]:.2f}, {trajectory[0, 1]:.2f})")
print(f"终点: ({trajectory[-1, 0]:.4f}, {trajectory[-1, 1]:.4f})")
print(f"目标: (0.0000, 0.0000)")

## 4. 随机梯度下降(SGD)

### 4.1 动机

**批量梯度下降(Batch GD)**的问题:
- 每次迭代需要计算**全部**样本的梯度
- 计算复杂度: $O(n)$,其中 $n$ 是样本数
- 大数据集($n$ 很大)时非常慢

**随机梯度下降(SGD)**的思想:
- 每次迭代只用**一个**随机样本
- 计算复杂度: $O(1)$
- 梯度是无偏估计: $\mathbb{E}[\nabla f_i(\mathbf{w})] = \nabla f(\mathbf{w})$

### 4.2 数学推导

目标函数(经验风险):
$$
f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n f_i(\mathbf{w})
$$

完整梯度:
$$
\nabla f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n \nabla f_i(\mathbf{w})
$$

SGD更新(随机选择索引 $i$):
$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla f_i(\mathbf{w})
$$

**无偏性**:
$$
\mathbb{E}_i[\nabla f_i(\mathbf{w})] = \frac{1}{n}\sum_{i=1}^n \nabla f_i(\mathbf{w}) = \nabla f(\mathbf{w})
$$

In [ ]:
# 模拟SGD: 给梯度添加噪声
def sgd_2d(lr, num_iters=50, x_init=(3.0, 3.0), noise_std=1.0):
    """带噪声的随机梯度下降"""
    x1, x2 = x_init
    trajectory = [(x1, x2)]
    
    for i in range(num_iters):
        g1, g2 = f_2d_grad(x1, x2)
        
        # 添加噪声模拟SGD
        g1 += np.random.randn() * noise_std
        g2 += np.random.randn() * noise_std
        
        x1 = x1 - lr * g1
        x2 = x2 - lr * g2
        trajectory.append((x1, x2))
    
    return trajectory

# 对比GD和SGD
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GD轨迹
traj_gd = np.array(gradient_descent_2d(lr=0.2, num_iters=50))
axes[0].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[0].plot(traj_gd[:, 0], traj_gd[:, 1], 'b-', linewidth=2, label='Batch GD')
axes[0].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[0].set_title('批量梯度下降 (平滑轨迹)')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# SGD轨迹
traj_sgd = np.array(sgd_2d(lr=0.1, num_iters=50))
axes[1].contour(X1, X2, Z, levels=15, cmap='viridis', alpha=0.4)
axes[1].plot(traj_sgd[:, 0], traj_sgd[:, 1], 'r-', linewidth=1.5, 
             alpha=0.7, label='SGD')
axes[1].scatter([0], [0], color='red', s=200, marker='*', zorder=5)
axes[1].set_title('随机梯度下降 (带噪声)')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print("\nSGD特点:")
print("1. 轨迹更加曲折(噪声导致)")
print("2. 即使接近最优,仍有振荡")
print("3. 噪声有助于逃离鞍点和局部最小值")
print("4. 需要学习率衰减来最终收敛")

### 4.3 SGD的优缺点

**优点**:
1. **计算高效**: 每次迭代$O(1)$复杂度
2. **内存友好**: 只需加载一个样本
3. **逃离鞍点**: 噪声有助于探索
4. **在线学习**: 可处理流式数据

**缺点**:
1. **方差大**: 梯度估计不准确
2. **不稳定**: 接近最优时仍振荡
3. **需要调参**: 学习率衰减策略
4. **无法并行**: 每次只用一个样本

## 5. 小批量SGD(Mini-batch SGD)

### 5.1 折中方案

**思想**: 每次用 $b$ 个样本的小批量
- $b = 1$: 标准SGD
- $b = n$: 批量GD
- $b \in (1, n)$: 小批量SGD(实践最常用)

**更新规则**:
$$
\mathbf{w} \leftarrow \mathbf{w} - \frac{\eta}{b} \sum_{i \in \mathcal{B}} \nabla f_i(\mathbf{w})
$$

其中 $\mathcal{B}$ 是随机采样的批量,大小为 $b$。

### 5.2 批量大小的影响

**小批量($b=32, 64, 128$)**:
- ✅ 计算效率高
- ✅ 梯度噪声有助于泛化
- ❌ 训练不稳定

**大批量($b=512, 1024, 2048$)**:
- ✅ 梯度估计更准确
- ✅ 训练稳定
- ✅ 更好的并行效率
- ❌ 可能陷入尖锐最小值(泛化差)
- ❌ 内存消耗大

**经验法则**: 从32或64开始,根据GPU内存和任务调整

In [ ]:
# 实际例子: 线性回归的不同优化方法对比
def generate_data(n_samples=1000, n_features=10):
    """生成线性回归数据"""
    torch.manual_seed(42)
    X = torch.randn(n_samples, n_features)
    true_w = torch.randn(n_features, 1)
    y = X @ true_w + torch.randn(n_samples, 1) * 0.1
    return X, y, true_w

def mse_loss(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean()

def train_comparison(X, y, method='batch', batch_size=32, lr=0.01, num_epochs=50):
    """训练并记录loss"""
    n_samples = X.shape[0]
    w = torch.randn(X.shape[1], 1, requires_grad=True)
    
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        n_batches = 0
        
        if method == 'batch':
            # 批量GD: 使用全部数据
            y_pred = X @ w
            loss = mse_loss(y_pred, y)
            loss.backward()
            
            with torch.no_grad():
                w -= lr * w.grad
                w.grad.zero_()
            
            losses.append(loss.item())
            
        elif method == 'sgd':
            # SGD: 每次一个样本
            indices = torch.randperm(n_samples)
            for i in indices[:min(50, n_samples)]:  # 限制迭代次数
                y_pred = X[i:i+1] @ w
                loss = mse_loss(y_pred, y[i:i+1])
                loss.backward()
                
                with torch.no_grad():
                    w -= lr * w.grad
                    w.grad.zero_()
                
                epoch_loss += loss.item()
                n_batches += 1
            
            losses.append(epoch_loss / n_batches if n_batches > 0 else 0)
            
        else:  # minibatch
            # 小批量SGD
            indices = torch.randperm(n_samples)
            for start_idx in range(0, n_samples, batch_size):
                batch_idx = indices[start_idx:start_idx + batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]
                
                y_pred = X_batch @ w
                loss = mse_loss(y_pred, y_batch)
                loss.backward()
                
                with torch.no_grad():
                    w -= lr * w.grad
                    w.grad.zero_()
                
                epoch_loss += loss.item()
                n_batches += 1
            
            losses.append(epoch_loss / n_batches)
    
    return losses

# 生成数据
X, y, true_w = generate_data(n_samples=1000, n_features=10)

# 比较三种方法
losses_batch = train_comparison(X, y, method='batch', lr=0.1, num_epochs=50)
losses_sgd = train_comparison(X, y, method='sgd', lr=0.01, num_epochs=50)
losses_minibatch = train_comparison(X, y, method='minibatch', batch_size=32, 
                                     lr=0.05, num_epochs=50)

# 可视化
plt.figure(figsize=(12, 6))
plt.plot(losses_batch, label='Batch GD (lr=0.1)', linewidth=2)
plt.plot(losses_sgd, label='SGD (lr=0.01)', linewidth=2, alpha=0.7)
plt.plot(losses_minibatch, label='Mini-batch SGD (batch=32, lr=0.05)', 
         linewidth=2, alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('优化方法对比 (线性回归)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print("\n观察:")
print(f"1. Batch GD: 最终loss = {losses_batch[-1]:.6f} (平滑但可能慢)")
print(f"2. SGD: 最终loss = {losses_sgd[-1]:.6f} (噪声大但探索性强)")
print(f"3. Mini-batch: 最终loss = {losses_minibatch[-1]:.6f} (平衡!)")

## 6. 小结

### 核心概念

1. **优化 vs 深度学习**:
   - 优化: 最小化训练误差
   - 深度学习: 最小化泛化误差
   - 需要正则化防止过拟合

2. **凸性**:
   - 凸函数: 局部最小值=全局最小值
   - 深度学习通常非凸,但局部最优通常足够好
   - 詹森不等式是重要工具

3. **梯度下降**:
   - 沿负梯度方向迭代更新
   - 学习率是关键超参数
   - 太小→慢,太大→发散

4. **SGD**:
   - 每次用一个样本,计算$O(1)$
   - 梯度是无偏估计
   - 噪声有助于探索和泛化

5. **Mini-batch SGD**:
   - 实践中的标准选择
   - 批量大小: 通常32-256
   - 平衡计算效率和梯度质量

### 优化方法对比

| 方法 | 批量大小 | 计算复杂度 | 梯度方差 | 并行性 | 适用场景 |
|------|----------|------------|----------|--------|----------|
| Batch GD | $n$ | $O(n)$ | 0 | ✅ | 小数据集 |
| SGD | 1 | $O(1)$ | 高 | ❌ | 在线学习 |
| Mini-batch | $b$ | $O(b)$ | 中 | ✅ | 通用 |

### 实践建议

1. **学习率**:
   - 从小开始(1e-3)
   - 观察loss曲线
   - 使用学习率衰减

2. **批量大小**:
   - 从32或64开始
   - GPU内存允许时增大
   - 大批量需要更大学习率

3. **收敛判断**:
   - 监控验证集loss
   - 早停(early stopping)
   - 梯度范数接近0

### 下一步

- **动量方法**: 加速收敛,平滑震荡
- **自适应学习率**: AdaGrad, RMSProp, Adam
- **学习率调度**: 指数衰减,余弦退火等

## 练习

1. **凸性判断**: 证明以下函数是否为凸函数:
   - $f(x) = |x|$
   - $f(x) = \max(0, x)$ (ReLU)
   - $f(x) = -\log(x)$ (负对数)

2. **学习率实验**: 在上面的线性回归例子中,测试学习率0.001, 0.01, 0.1, 1.0的效果。

3. **批量大小**: 比较批量大小1, 8, 32, 128, 1024对训练速度和最终性能的影响。

4. **非凸优化**: 实现f(x) = x⁴ - 3x³ + 2的梯度下降,从不同起点出发,观察是否收敛到同一点。

5. **鞍点**: 对于f(x, y) = x² - y²,从(1, 1)出发进行梯度下降,观察是否能逃离鞍点(0, 0)。